# Day 5-01｜投籃分析 Pipeline Runner

> Python「黑客松」實戰分析籃球運動數據  
> Notebook 以初學 Python 學員為主：先觀察、再改變少量變數、最後才串接。

目標：把 Day 4 的 pose angle 與 ball track 結果讀回來，產生一份簡單 summary。

In [ ]:
from pathlib import Path
import sys

# 在 Colab 中先掛載 Drive；本機執行時會自動略過。
try:
    from google.colab import drive

    drive.mount("/content/drive")
except Exception:
    pass

COURSE_ROOT = Path("/content/drive/MyDrive/basketball_hackathon/course")
if not COURSE_ROOT.exists():
    # 本機或 zip 解壓後執行時，從目前資料夾往上找。
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "assets").exists():
            COURSE_ROOT = candidate
            break

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

print("COURSE_ROOT =", COURSE_ROOT)
print("assets =", COURSE_ROOT / "assets")

In [ ]:
import json
import pandas as pd
from src.shooting_utils import synthetic_pose_sequence, estimate_release_frame

RESULTS = COURSE_ROOT / "assets" / "results"
pose_csv = RESULTS / "d4_02_pose_angles.csv"
ball_csv = RESULTS / "d4_03_ball_track.csv"

if pose_csv.exists():
    pose_df = pd.read_csv(pose_csv)
else:
    pose_df = synthetic_pose_sequence(n=80)
    print("找不到 pose csv，使用合成 pose。")

if ball_csv.exists():
    ball_df = pd.read_csv(ball_csv)
else:
    ball_df = pd.DataFrame()
    print("找不到 ball csv；summary 會略過 ball。")

pose_df.head()

In [ ]:
summary = {
    "pose_frames": int(len(pose_df)),
    "max_elbow_angle": float(pose_df["elbow_angle"].max())
    if "elbow_angle" in pose_df
    else None,
    "min_elbow_angle": float(pose_df["elbow_angle"].min())
    if "elbow_angle" in pose_df
    else None,
    "max_knee_angle": float(pose_df["knee_angle"].max())
    if "knee_angle" in pose_df
    else None,
    "ball_points": int(len(ball_df)),
    "estimated_release_frame": estimate_release_frame(ball_df)
    if len(ball_df)
    else None,
}
summary

In [ ]:
out_json = RESULTS / "d5_01_shooting_summary.json"
out_json.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", out_json)

這份 summary 不是正式運動科學評估，只是讓學生知道如何把影像分析轉成可報告的數字。